# Question Answering

Mirrors the structure of the LangChain "Chat with Your Data" question-answering lecture (`05_question_answering.ipynb`), applied to our real Bahraini legal corpus.

## Overview

Recall the RAG workflow so far: we scraped and cleaned three Bahraini legal sources, split them into passages, embedded them, and built a retriever on top of the v2 vector store (see `05_retrieval_langchain.ipynb`). This notebook covers the next step: turning retrieved passages into an actual, cited Arabic answer.

In [ ]:
!pip install -q langchain-huggingface langchain-chroma langchain-core langchain-text-splitters langchain-community langchain-classic langchain-openai sentence-transformers transformers chromadb


In [ ]:
import os
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")


In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from google.colab import drive
import torch
drive.mount("/content/drive")

persist_directory = "/content/drive/MyDrive/law_chatbot_chroma_v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": device})
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)
print(vectordb._collection.count())


In [ ]:
question = "هل يجوز لصاحب العمل فصل عامل تغيب عن العمل دون انذار؟"
docs = vectordb.similarity_search(question, k=3)
len(docs)

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    temperature=0,
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

### RetrievalQA chain — default

In [ ]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(llm, retriever=vectordb.as_retriever(search_kwargs={"k": 2}))
result = qa_chain.invoke({"query": question})
result["result"]

### Prompt — Arabic, with strict citation rules

In [ ]:
from langchain_core.prompts import PromptTemplate

SYSTEM_TEMPLATE = """انت مساعد قانوني متخصص في القانون البحريني. استخدم المقاطع القانونية التالية فقط للاجابة على السؤال في نهاية النص.

قواعد صارمة يجب اتباعها:
- استند فقط الى النصوص المرفقة، ولا تخترع اي معلومة غير موجودة فيها.
- اذا لم تكن الاجابة موجودة في النصوص المرفقة، صرح بذلك بوضوح ولا تخمن.
- اذكر المصدر الدقيق لكل معلومة (رقم المادة او رقم القضية).
- اذا استندت الاجابة الى اكثر من قانون او حكم، اذكرهم جميعا.

النصوص القانونية:
{context}

السؤال: {question}

الاجابة القانونية المدعومة بالمصادر:"""

QA_CHAIN_PROMPT = PromptTemplate.from_template(SYSTEM_TEMPLATE)

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 2}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT},
)

result = qa_chain.invoke({"query": question})
print(result["result"])

In [ ]:
for doc in result["source_documents"]:
    print(doc.metadata)

### Chain types — `stuff` vs `map_reduce` vs `refine`

In [ ]:
import time

qa_chain_map_reduce = RetrievalQA.from_chain_type(
    llm, retriever=vectordb.as_retriever(search_kwargs={"k": 2}), chain_type="map_reduce"
)

print("--- map_reduce ---")
print(qa_chain_map_reduce.invoke({"query": question})["result"])

If you want to experiment with the **LangSmith** platform to trace and inspect these chains (this mirrors the lecture's optional LangSmith step — it is off by default and nothing here requires it):

* Go to [LangSmith](https://www.langchain.com/langsmith) and sign up
* Create an API key from your account's settings
* Paste it below and uncomment the cell


In [ ]:
# import os
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
# os.environ["LANGCHAIN_API_KEY"] = "..."  # replace with your own LangSmith key
# os.environ["LANGCHAIN_PROJECT"] = "capital-legal-base"


In [ ]:
qa_chain_refine = RetrievalQA.from_chain_type(
    llm, retriever=vectordb.as_retriever(search_kwargs={"k": 2}), chain_type="refine"
)

time.sleep(15)

print("--- refine ---")
print(qa_chain_refine.invoke({"query": question})["result"])

### RetrievalQA limitations

QA fails to preserve conversational history.

In [ ]:
follow_up = "لماذا يشترط القانون الانذار قبل الفصل؟"
result = qa_chain.invoke({"query": follow_up})
result["result"]